# Chapter 6: Volatility in code

Normal-distribution probabilities, scaling volatility for time, checking observed price changes, normal vs lognormal values, implied volatility and the gold-option tables.
Notes: [[chapters/06-volatility]]

Only the standard library and matplotlib are used. `math.erf` gives the normal cumulative probability.

In [ ]:
import math
import matplotlib.pyplot as plt
from math import sqrt, exp, log, pi, erf

def N(x):
    """Standard normal cumulative probability: chance a value is below x standard deviations from the mean."""
    return 0.5 * (1 + erf(x / sqrt(2)))

def within(k):
    """Chance of being within +/- k standard deviations of the mean."""
    return N(k) - N(-k)

for k in (1, 2, 3):
    print(f"within +/-{k} st. dev.: {within(k):.4f}")

within +/-1 st. dev.: 0.6827
within +/-2 st. dev.: 0.9545
within +/-3 st. dev.: 0.9973


## The pinball maze: troughs 0 to 15, mean 7.5, standard deviation 3.0

In [2]:
mean, sd = 7.5, 3.0
print("1 st. dev. range:", mean - sd, "to", mean + sd, "-> troughs 5 to 10")
print("P(trough < 5 or > 10) approx 1 - 0.683 =", round(1 - within(1), 3))
print("P(beyond 2 st. dev.) =", round(1 - within(2), 4), " (about 1 in", round(1 / (1 - within(2))), ")")
print("P(troughs 14 or 15 only) approx half of that =", round((1 - within(2)) / 2, 4), " (about 1 in", round(2 / (1 - within(2))), ")")
print("A 30-to-1 payout is fair only if the chance is 1/31 =", round(1 / 31, 4), "-> bet is bad")
print()
# our own check: a real 15-coin-flip has mean 7.5 but a smaller standard deviation
print("15 fair coin flips: sd =", round(sqrt(15 * 0.5 * 0.5), 3), "(not 3.0, so the book's maze is illustrative)")

1 st. dev. range: 4.5 to 10.5 -> troughs 5 to 10
P(trough < 5 or > 10) approx 1 - 0.683 = 0.317
P(beyond 2 st. dev.) = 0.0455  (about 1 in 22 )
P(troughs 14 or 15 only) approx half of that = 0.0228  (about 1 in 44 )
A 30-to-1 payout is fair only if the chance is 1/31 = 0.0323 -> bet is bad

15 fair coin flips: sd = 1.936 (not 3.0, so the book's maze is illustrative)


## Volatility as a standard deviation
Forward 100, volatility 20%; and a stock at 100, 8% interest, 1 year, no dividend.

In [3]:
vol = 0.20
print("Forward 100:")
for k in (1, 2, 3):
    print(f"  {k} st. dev.: {100 * (1 - k * vol):.0f} to {100 * (1 + k * vol):.0f}   ({within(k):.1%})")

F = 100 * (1 + 0.08 * 1)
one_sd = vol * F
print()
print("Stock 100, r = 8%, no dividend: forward =", F, " one st. dev. =", one_sd)
for k in (1, 2, 3):
    print(f"  {k} st. dev.: {F - k * one_sd:.2f} to {F + k * one_sd:.2f}")

# could a price of 35 after a year be consistent with 20% volatility? (forward 100)
z = (35 - 100) / (100 * vol)
print()
print(f"Price 35 is {z:.2f} st. dev. from the mean; chance of being that low or lower = {N(z):.5f} (about 1 in {round(1 / N(z))})")
print("15 heads in 15 flips: 1 in", 2 ** 15)

Forward 100:
  1 st. dev.: 80 to 120   (68.3%)
  2 st. dev.: 60 to 140   (95.4%)
  3 st. dev.: 40 to 160   (99.7%)

Stock 100, r = 8%, no dividend: forward = 108.0  one st. dev. = 21.6
  1 st. dev.: 86.40 to 129.60
  2 st. dev.: 64.80 to 151.20
  3 st. dev.: 43.20 to 172.80

Price 35 is -3.25 st. dev. from the mean; chance of being that low or lower = 0.00058 (about 1 in 1733)
15 heads in 15 flips: 1 in 32768


## Scaling volatility for time: multiply by the square root of the time in years

In [4]:
def scaled_vol(annual, t_years):
    return annual * sqrt(t_years)

print("sqrt(256) =", sqrt(256), "  sqrt(52) =", round(sqrt(52), 3))

def move(price, annual_vol, periods_per_year):
    """One standard deviation price change over one period."""
    return price * annual_vol / sqrt(periods_per_year)

print()
print("Contract at 100, volatility 20%:")
print("  daily  1 st. dev. =", round(move(100, .20, 256), 2), " 2 st. dev. =", round(2 * move(100, .20, 256), 2))
print("  weekly 1 st. dev. =", round(move(100, .20, 52), 2), " 2 st. dev. =", round(2 * move(100, .20, 52), 2))

print()
print("Stock at 45, volatility 37%:")
for name, n in (("day", 256), ("week", 52)):
    m = move(45, .37, n)
    print(f"  one {name}: 1 st. dev. = {m:.2f} -> {45 - m:.2f} to {45 + m:.2f};  2 st. dev. -> {45 - 2 * m:.2f} to {45 + 2 * m:.2f}")

print()
print("Eurodollar futures at 93.00, rate volatility 26% (use the RATE, 100 - 93 = 7):")
print("  daily  =", round((100 - 93) * .26 / 16, 2))
print("  weekly =", round((100 - 93) * .26 / sqrt(52), 2))

sqrt(256) = 16.0   sqrt(52) = 7.211

Contract at 100, volatility 20%:
  daily  1 st. dev. = 1.25  2 st. dev. = 2.5
  weekly 1 st. dev. = 2.77  2 st. dev. = 5.55

Stock at 45, volatility 37%:
  one day: 1 st. dev. = 1.04 -> 43.96 to 46.04;  2 st. dev. -> 42.92 to 47.08
  one week: 1 st. dev. = 2.31 -> 42.69 to 47.31;  2 st. dev. -> 40.38 to 49.62

Eurodollar futures at 93.00, rate volatility 26% (use the RATE, 100 - 93 = 7):
  daily  = 0.11
  weekly = 0.25


## Are five daily changes consistent with 37% volatility?
Changes: +0.98, -0.65, -0.70, +0.25, -0.85 on a 45 stock. The book says the realized volatility is 27.8%. The footnote says volatility is normally computed from **logarithmic** price changes, so we do that.

In [5]:
changes = [0.98, -0.65, -0.70, 0.25, -0.85]
prices = [45.0]
for c in changes:
    prices.append(prices[-1] + c)
log_returns = [log(prices[i + 1] / prices[i]) for i in range(len(changes))]

def sample_sd(xs):
    m = sum(xs) / len(xs)
    return sqrt(sum((x - m) ** 2 for x in xs) / (len(xs) - 1))

print("one st. dev. daily move at 37%:", round(move(45, .37, 256), 2))
print("moves bigger than that:", sum(abs(c) > move(45, .37, 256) for c in changes), "of", len(changes), "(expect about 1 in 3)")
print("realized volatility (log changes, sample sd, x16):", round(sample_sd(log_returns) * 16 * 100, 1), "%")

one st. dev. daily move at 37%: 1.04
moves bigger than that: 0 of 5 (expect about 1 in 3)
realized volatility (log changes, sample sd, x16): 27.8 %


## Compounding frequency (the book's tables), and the exponential

In [6]:
print(f"{'payments per year':>18}{'+12%':>12}{'-12%':>12}")
for n in (1, 2, 4, 12, 52, 365):
    print(f"{n:>18}{1000 * (1 + .12 / n) ** n:>12.2f}{1000 * (1 - .12 / n) ** n:>12.2f}")
print(f"{'continuous':>18}{1000 * exp(.12):>12.2f}{1000 * exp(-.12):>12.2f}")
print()
print("Continuous gain +12%:", round(1000 * exp(.12) - 1000, 2), "  continuous loss -12%:", round(1000 - 1000 * exp(-.12), 2))

 payments per year        +12%        -12%
                 1     1120.00      880.00
                 2     1123.60      883.60
                 4     1125.51      885.29
                12     1126.83      886.38
                52     1127.34      886.80
               365     1127.47      886.90
        continuous     1127.50      886.92

Continuous gain +12%: 127.5   continuous loss -12%: 113.08


## Pricing functions (Black form with a forward price)
Call and put values from a forward price `F`, using the normal cumulative `N`. These are the lognormal-model values the book quotes. The formula itself is covered in a later chapter; here it is a tool.

In [7]:
def black(F, K, T, vol, r, kind):
    """Option value from forward F, strike K, years T, volatility vol, interest r (continuous discounting)."""
    d1 = (log(F / K) + 0.5 * vol * vol * T) / (vol * sqrt(T))
    d2 = d1 - vol * sqrt(T)
    df = exp(-r * T)
    if kind == "call":
        return df * (F * N(d1) - K * N(d2))
    return df * (K * N(-d2) - F * N(-d1))

def stock_option(S, K, T, vol, r, kind):
    """Stock with no dividend: the forward price is S grown at the interest rate."""
    return black(S * exp(r * T), K, T, vol, r, kind)

def normal_model(F, K, T, vol, kind):
    """Value if the price at expiration is normal with mean F and standard deviation vol*F*sqrt(T). No discounting."""
    s = vol * F * sqrt(T)
    d = (F - K) / s if kind == "call" else (K - F) / s
    pdf = exp(-d * d / 2) / sqrt(2 * pi)
    return (F - K) * N(d) + s * pdf if kind == "call" else (K - F) * N(d) + s * pdf

## Normal vs lognormal: 90 put and 110 call, forward 100, six months, volatility 30%
Book: normal 4.37 and 4.37; lognormal 4.00 and 4.74. Our numbers differ slightly (the book does not give every convention it used) but show the same pattern.

In [8]:
F, T, vol = 100, 0.5, 0.30
print(f"{'':<10}{'normal':>10}{'lognormal':>12}")
print(f"{'90 put':<10}{normal_model(F, 90, T, vol, 'put'):>10.2f}{black(F, 90, T, vol, 0, 'put'):>12.2f}")
print(f"{'110 call':<10}{normal_model(F, 110, T, vol, 'call'):>10.2f}{black(F, 110, T, vol, 0, 'call'):>12.2f}")

              normal   lognormal
90 put          4.39        3.99
110 call        4.39        4.75


## Implied volatility: the 105 call
Stock 98.50, three months, 6% interest, no dividend. Our volatility estimate is 25%; the market price is 3.60. We run the model backwards by bisection (keep halving a range until the model value matches the price).

In [9]:
S, K, T, r = 98.50, 105, 0.25, 0.06

def implied_vol(price, value_fn, lo=0.01, hi=3.0):
    """Find the volatility at which value_fn(vol) equals price. value_fn must rise with volatility."""
    for _ in range(100):
        mid = (lo + hi) / 2
        if value_fn(mid) < price:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

theo = stock_option(S, K, T, 0.25, r, "call")
iv = implied_vol(3.60, lambda v: stock_option(S, K, T, v, r, "call"))
print(f"Theoretical value at 25% volatility: {theo:.2f}   (book: 2.94)")
print(f"Implied volatility at price 3.60:    {iv:.2%}  (book: 28.50%)")
print(f"Mispricing in points: {3.60 - theo:.2f}   in volatility points: {(iv - 0.25) * 100:.2f}   (book: 0.66 and 3.50)")

# what else could explain the 3.60?  (solve for S or r holding volatility at 25%)
def solve(f, target, lo, hi):
    for _ in range(100):
        mid = (lo + hi) / 2
        if f(mid) < target: lo = mid
        else: hi = mid
    return (lo + hi) / 2

print(f"Stock price needed for 3.60 at 25% vol: {solve(lambda s: stock_option(s, K, T, .25, r, 'call'), 3.60, 90, 110):.2f}   (book: 100.16)")
print(f"Interest rate needed for 3.60 at 25% vol: {solve(lambda x: stock_option(S, K, T, .25, x, 'call'), 3.60, 0, .5):.2%}   (book: 13.30%)")

# a stale price: the 3.60 trade was when the stock was 99.25
print(f"If the stock was really 99.25: implied volatility {implied_vol(3.60, lambda v: stock_option(99.25, K, T, v, r, 'call')):.2%}   (book: 26.95%)")

# compare the 100 call at 5.40
iv100 = implied_vol(5.40, lambda v: stock_option(S, 100, T, v, r, "call"))
print(f"100 call at 5.40: implied volatility {iv100:.2%}   (book: 27.51%) -> cheaper in volatility terms than the 105 call at {iv:.2%}")

Theoretical value at 25% volatility: 2.94   (book: 2.94)
Implied volatility at price 3.60:    28.50%  (book: 28.50%)
Mispricing in points: 0.66   in volatility points: 3.50   (book: 0.66 and 3.50)
Stock price needed for 3.60 at 25% vol: 100.16   (book: 100.16)
Interest rate needed for 3.60 at 25% vol: 13.29%   (book: 13.30%)
If the stock was really 99.25: implied volatility 26.93%   (book: 26.95%)
100 call at 5.40: implied volatility 27.53%   (book: 27.51%) -> cheaper in volatility terms than the 105 call at 28.50%


## Gold options (July 31, 2012): what happens when volatility moves from 14% to 18%?
October futures 1612.4, 56 days, interest 0.50%.

In [10]:
def gold_table(F, days, r=0.005):
    T = days / 365
    rows = []
    for K in (1400, 1500, 1600, 1700, 1800):
        row = {"K": K}
        for kind in ("call", "put"):
            v14 = black(F, K, T, .14, r, kind)
            v18 = black(F, K, T, .18, r, kind)
            row[kind] = (v14, v18, v18 - v14, 100 * (v18 / v14 - 1))
        rows.append(row)
    return rows

oct_rows = gold_table(1612.4, 56)
dec_rows = gold_table(1614.6, 119)
for name, rows in (("October", oct_rows), ("December", dec_rows)):
    print(name)
    print(f"{'K':>6} | {'call 14%':>9}{'call 18%':>9}{'chg':>7}{'%chg':>7} | {'put 14%':>9}{'put 18%':>9}{'chg':>7}{'%chg':>7}")
    for row in rows:
        c, p = row["call"], row["put"]
        print(f"{row['K']:>6} | {c[0]:>9.2f}{c[1]:>9.2f}{c[2]:>7.2f}{c[3]:>7.0f} | {p[0]:>9.2f}{p[1]:>9.2f}{p[2]:>7.2f}{p[3]:>7.0f}")

October
     K |  call 14% call 18%    chg   %chg |   put 14%  put 18%    chg   %chg
  1400 |    212.37   213.13   0.76      0 |      0.13     0.89   0.76    584
  1500 |    116.05   121.01   4.97      4 |      3.74     8.70   4.97    133
  1600 |     41.65    51.60   9.95     24 |     29.26    39.21   9.95     34
  1700 |      8.08    15.28   7.21     89 |     95.61   102.82   7.21      8
  1800 |      0.78     3.05   2.28    293 |    188.23   190.51   2.28      1
December
     K |  call 14% call 18%    chg   %chg |   put 14%  put 18%    chg   %chg
  1400 |    216.03   220.06   4.03      2 |      1.78     5.81   4.03    226
  1500 |    126.41   136.59  10.18      8 |     12.00    22.18  10.18     85
  1600 |     58.78    73.31  14.53     25 |     44.20    58.74  14.53     33
  1700 |     20.71    33.53  12.81     62 |    105.97   118.79  12.81     12
  1800 |      5.44    13.03   7.59    139 |    190.54   198.13   7.59      4


## Figures

In [ ]:
#import os
# uncomment if you want to replace images
os.makedirs("../images", exist_ok=True)
import numpy as np

# 1. Same strike, three distributions: more spread, more of the curve beyond the strike
x = np.linspace(40, 160, 600)
fig, ax = plt.subplots(figsize=(9, 4.5))
strike = 110
for sd_, col, name in ((8, "#3182bd", "low volatility"), (14, "#31a354", "moderate volatility"), (24, "#e6550d", "high volatility")):
    pdf = np.exp(-0.5 * ((x - 100) / sd_) ** 2) / (sd_ * sqrt(2 * pi))
    ax.plot(x, pdf, color=col, label=f"{name} (st. dev. {sd_})")
    mask = x >= strike
    ax.fill_between(x[mask], pdf[mask], color=col, alpha=0.25)
    print(f"{name}: chance above {strike} = {1 - N((strike - 100) / sd_):.1%}")
ax.axvline(strike, color="k", ls="--"); ax.text(strike + 1, ax.get_ylim()[1] * 0.9, "exercise price 110")
ax.axvline(100, color="grey", ls=":"); ax.text(101, ax.get_ylim()[1] * 0.97, "current price 100")
ax.set_xlabel("Underlying price at expiration"); ax.set_yticks([])
ax.set_title("Wider distribution, more area beyond the exercise price (shaded)")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout(); fig.savefig("../images/ch06-spread-and-call-value.png", dpi=130); plt.close(fig)

# 2. 68 / 95 / 99.7
z = np.linspace(-4, 4, 600)
pdf = np.exp(-z ** 2 / 2) / sqrt(2 * pi)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(z, pdf, color="k")
for k, col in ((3, "#deebf7"), (2, "#9ecae1"), (1, "#3182bd")):
    m = np.abs(z) <= k
    ax.fill_between(z[m], pdf[m], color=col, label=f"within {k} st. dev.: {within(k):.1%}")
ax.set_xticks(range(-4, 5)); ax.set_xticklabels(["-4", "-3σ", "-2σ", "-1σ", "mean", "+1σ", "+2σ", "+3σ", "+4"])
ax.set_yticks([]); ax.legend(loc="upper left")
ax.set_title("The normal distribution: 68.3% / 95.4% / 99.7% rule")
fig.tight_layout(); fig.savefig("../images/ch06-normal-68-95-997.png", dpi=130); plt.close(fig)

# 3. normal vs lognormal, forward 100, 6 months, vol 30%
F, T, vol = 100, 0.5, 0.30
s = vol * sqrt(T)
x = np.linspace(1, 220, 800)
normal_pdf = np.exp(-0.5 * ((x - F) / (F * s)) ** 2) / (F * s * sqrt(2 * pi))
mu = log(F) - s * s / 2
ln_pdf = np.exp(-0.5 * ((np.log(x) - mu) / s) ** 2) / (x * s * sqrt(2 * pi))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, normal_pdf, color="#3182bd", label="normal (symmetric, can go below 0)")
ax.plot(x, ln_pdf, color="#e6550d", label="lognormal (skewed up, never below 0)")
ax.axvline(F, color="grey", ls=":"); ax.text(F + 1, ax.get_ylim()[1] * 0.97, "mean = forward 100")
mode = exp(mu - s * s)
ax.axvline(mode, color="#e6550d", ls=":"); ax.text(mode - 2, ax.get_ylim()[1] * 0.85, f"lognormal peak {mode:.0f}", ha="right", color="#e6550d")
ax.set_xlim(0, 220); ax.set_yticks([]); ax.set_xlabel("Underlying price at expiration")
ax.set_title("Normal vs lognormal price distributions (same mean, volatility 30%, six months)")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout(); fig.savefig("../images/ch06-normal-vs-lognormal.png", dpi=130); plt.close(fig)

# 4. implied volatility: model value against volatility for the 105 call
vols = np.linspace(0.05, 0.50, 200)
vals = [stock_option(98.5, 105, 0.25, v, 0.06, "call") for v in vols]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(vols * 100, vals, color="k")
ax.axhline(3.60, color="#e6550d", ls="--"); ax.text(6, 3.7, "market price 3.60", color="#e6550d")
ax.plot([25], [theo], "o", color="#3182bd"); ax.annotate(f"our 25% -> value {theo:.2f}", (25, theo), (27, theo - 0.9), color="#3182bd")
ax.plot([iv * 100], [3.60], "o", color="#e6550d"); ax.annotate(f"implied volatility {iv:.2%}", (iv * 100, 3.60), (30, 4.6), color="#e6550d")
ax.set_xlabel("Volatility input (%)"); ax.set_ylabel("Theoretical value of the 105 call")
ax.set_title("Implied volatility: run the model backwards until value = price")
fig.tight_layout(); fig.savefig("../images/ch06-implied-volatility.png", dpi=130); plt.close(fig)

# 5. gold: change in value when volatility goes 14% -> 18%
ks = [r["K"] for r in oct_rows]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
w = 0.38
idx = np.arange(len(ks))
axes[0].bar(idx - w / 2, [r["call"][2] for r in oct_rows], w, label="October calls", color="#9ecae1")
axes[0].bar(idx + w / 2, [r["call"][2] for r in dec_rows], w, label="December calls", color="#3182bd")
axes[0].set_title("Change in value, points (14% to 18% volatility)")
axes[1].bar(idx - w / 2, [r["call"][3] for r in oct_rows], w, label="October calls", color="#fdd0a2")
axes[1].bar(idx + w / 2, [r["call"][3] for r in dec_rows], w, label="December calls", color="#e6550d")
axes[1].set_title("Change in value, percent")
for a in axes:
    a.set_xticks(idx); a.set_xticklabels(ks); a.set_xlabel("Exercise price (futures about 1613)"); a.legend(fontsize=9)
fig.suptitle("Gold calls: at-the-money moves most in points, far out-of-the-money most in percent; longer-dated moves more", fontsize=10)
fig.tight_layout(); fig.savefig("../images/ch06-gold-volatility-sensitivity.png", dpi=130); plt.close(fig)
print("saved five figures to ../images")

low volatility: chance above 110 = 10.6%
moderate volatility: chance above 110 = 23.8%
high volatility: chance above 110 = 33.8%
saved five figures to ../images
